# Text-to-SQL fine-tuning notebook

This notebook is the GPU-first entry point for the project. It installs dependencies, pulls the repo, checks the runtime, evaluates the baseline, fine-tunes the model, and reports the before/after metrics.

In [ ]:
# Mount Drive first because /content is temporary.
from google.colab import drive
import os
import subprocess
import sys

DRIVE_ROOT = "/content/drive/MyDrive/text2sql-finetune"
drive.mount("/content/drive")
assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted. Stop and mount Drive before continuing."
os.makedirs(DRIVE_ROOT, exist_ok=True)

!pip install --quiet unsloth
!pip install --quiet datasets transformers trl peft bitsandbytes accelerate

REPO_URL = "https://github.com/pmmeenakshi/text2sql-finetune.git"
REPO_PATH = "/content/repo"
if os.path.isdir(REPO_PATH):
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.chdir(REPO_PATH)
print("Drive root:", DRIVE_ROOT)
print("Repo ready for import from:", REPO_PATH)

In [ ]:
import torch
import os

from config import DRIVE_ROOT

assert os.path.isdir("/content/drive/MyDrive"), "Drive is not mounted. Stop before running GPU work."
print('Drive output root:', DRIVE_ROOT)
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Switch the notebook runtime to a GPU-backed option in Colab or Kaggle.')

In [ ]:
from config import TRAIN_SUBSET
from data import load_sql_dataset, render_prompt

train_ds, test_ds = load_sql_dataset(train_subset=TRAIN_SUBSET)
print('Train size:', len(train_ds))
print('Test size:', len(test_ds))
row = train_ds[0]
print('Example row keys:', list(row.keys()))
print('Example question:', row['question'])
print('Example schema:', row['context'][:200])

In [ ]:
from train import load_base_model
from evaluate import run_evaluation, run_harness_checks
from config import BASE_PREDICTIONS_PATH, EVAL_SAMPLE_SIZE

model, tokenizer = load_base_model()
harness_results = run_harness_checks(test_ds, n=EVAL_SAMPLE_SIZE)
print('Harness results:', harness_results)
if harness_results['identity_accuracy'] != 1.0:
    raise RuntimeError('Harness identity check failed. Stop before recording metrics.')

base_metrics = run_evaluation(
    model,
    tokenizer,
    test_ds,
    n=EVAL_SAMPLE_SIZE,
    predictions_path=BASE_PREDICTIONS_PATH,
)
print('BASELINE metrics:', base_metrics)

In [ ]:
from train import attach_lora, train_model
from config import ADAPTER_DIR, CHECKPOINT_DIR
import os

model = attach_lora(model)
checkpoint_paths = [
    os.path.join(CHECKPOINT_DIR, name)
    for name in os.listdir(CHECKPOINT_DIR)
    if name.startswith("checkpoint-")
    and os.path.isdir(os.path.join(CHECKPOINT_DIR, name))
]
resume_checkpoint = max(
    checkpoint_paths,
    key=lambda path: int(os.path.basename(path).split("-")[-1]),
) if checkpoint_paths else None
print("Resuming from:", resume_checkpoint or "start")
trainer = train_model(
    model,
    tokenizer,
    train_ds,
    resume_from_checkpoint=resume_checkpoint,
)

# Save immediately after training, before evaluation can consume the session.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
saved_files = []
for root, _, files in os.walk(ADAPTER_DIR):
    for filename in files:
        file_path = os.path.join(root, filename)
        saved_files.append((file_path, os.path.getsize(file_path)))
assert saved_files, 'Adapter directory is empty; stop before evaluation.'
for file_path, file_size in saved_files:
    print(file_path, file_size, 'bytes')
print('LoRA training finished.')

In [ ]:
import json
from evaluate import run_evaluation
from config import EVAL_SAMPLE_SIZE, FINE_TUNED_PREDICTIONS_PATH, RESULTS_PATH

fine_tuned_metrics = run_evaluation(
    model,
    tokenizer,
    test_ds,
    n=EVAL_SAMPLE_SIZE,
    predictions_path=FINE_TUNED_PREDICTIONS_PATH,
)
print('FINE-TUNED metrics:', fine_tuned_metrics)

final_results = {
    'baseline': base_metrics,
    'fine_tuned': fine_tuned_metrics,
    'harness': harness_results,
}
with open(RESULTS_PATH, 'w', encoding='utf-8') as results_file:
    json.dump(final_results, results_file, indent=2)

print('\nFinal comparison')
print('Model | Exact Match | Execution Accuracy | Informative examples | n')
print('Base | {exact_match:.2%} | {execution_accuracy:.2%} | {informative_execution_examples} | {total_examples}'.format(**base_metrics))
print('Fine-tuned | {exact_match:.2%} | {execution_accuracy:.2%} | {informative_execution_examples} | {total_examples}'.format(**fine_tuned_metrics))
print('Results saved to:', RESULTS_PATH)

This final section saves the adapter and shows a single inference example using a hand-written schema and question.

In [ ]:
from config import ADAPTER_DIR

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter to:', ADAPTER_DIR)

question = "List all users with names starting with A."
schema = "CREATE TABLE users (id INTEGER, name TEXT);"
prompt = render_prompt(question, schema)
print('\nInference prompt preview:')
print(prompt)